### Memory

In [44]:
from langchain_core.chat_history import BaseChatMessageHistory

# 채팅 기록을 저장할 클래스 정의
# BaseCahtMessageHistory를 상속받는다.
class InMemoryHistory(BaseChatMessageHistory):
    def __init__(self):
        self.messages = [] # 메세지를 저장할 빈 리스트 생성

    def add_messages(self, messages):
        # 메세지 리스트가 들어오면 기존 리스트에 추가
        self.messages.extend(messages)

    def clear(self):
        # 기록 초기화
        self.messages = []

# 세션(사용자)별로 기록을 관리할 딕셔너리 생성
# store = {'user1' : History 객체, 'user2' : History 객체, ...}
store = {}

# 세션 ID를 받아 해당 세션의 History 객체를 반환하는 함수
def get_by_session_id(session_id):
    if session_id not in store:
        # 새로운 InMemoryHistory 객체를 만들어서 store에 저장
        store[session_id] = InMemoryHistory()
    # 해당하는 session_id 기록 객체를 반환
    return store[session_id]

In [45]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory # Runnable은 chain을 의미.

# 프롬프트 템플릿 생성
prompts = ChatPromptTemplate.from_messages([
    ('system', '너는 {skill}을 잘하는 도우미야'), # 시스템 메세지 (페르소나 설정)
    MessagesPlaceholder(variable_name="history"), # 대화 기록이 주입될 위치
    ('human', '{query}'), # 사용자 질문
])

# 모델 생성
model = ChatOpenAI(model='gpt-5-nano', temperature=0.8)

# 체인 생성
chain = prompts | model

# 메모리 기능 장착한 체인 생성(RunnableWithMessageHistory)
# 기존 chain을 감싸서, 대화 기록을 자동으로 읽고 쓰는 기능 추가
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history=get_by_session_id, # 세션 ID로 기록을 찾아오는 함수 연결
    input_messages_key='query', # 사용자 입력이 들어오는 변수명
    history_messages_key='history' # 프롬프트에서 기록이 들어갈 변수명 (Placeholder 이름과 일치해야 함)
)

In [46]:
# 첫 번째 질문
response = chain_with_history.invoke(
    {'skill':'math', 'query':'철수는 강아지를 세마리 키우고 있습니다.'},
    config={'configurable':{'session_id':'abc'}} # session_id를 'abc'로 설정
)

print(response.content)

도와드릴게요! 현재 문장은 상황 서술인데, 수학 문제로 활용할 수 있는 예를 몇 가지 바로 보여드릴게요. 원하시는 문제나 데이터를 주시면 바로 계산해 드리겠습니다.

예시 문제들
- 조합: 세 마리 중 두 마리를 골라 팀을 만든다면 경우의 수는?
  3C2 = 3가지

- 순열: 세 마리의 강아지를 한 줄로 나열하는 방법 수는?
  3! = 6가지

- 간식 선택 예시: 강아지 3마리가 서로 다른 간식을 고르는 경우의 수를 2종류로 가정하면?
  2^3 = 8가지

- 무게 합 예시: 세 강아지의 몸무게가 각각 4 kg, 5 kg, 6 kg일 때 전체 무게는?
  합 = 4 + 5 + 6 = 15 kg
  평균 무게 = 15 / 3 = 5 kg

- 비율/배분 예시: 세 강아지에게 간식 총량을 똑같이 나누면 각 강아지는 몇 그램이 받나? (총량이 주어지면 계산)

원하시는 문제 유형이나 실제 데이터(예: 각 강아지의 무게, 간식 종류 수 등)를 알려주시면 바로 계산해 드리겠습니다.


In [47]:
# 두 번째 질문
response2 = chain_with_history.invoke(
    {'skill':'math','query':'영회가 고양이를 두 마리 키우고 있습니다.'},
    config={'configurable': {'session_id' : 'abc'}}
)

print(response2.content)

좋아요! 영회가 고양이 두 마리인 상황으로 바로 활용할 수 있는 수학 예시를 몇 가지 정리해 드릴게요.

예시 문제 및 해설
1) 조합
- 두 고양이 중 한 마리를 고르는 경우의 수: 2C1 = 2
- 예: 어떤 보호자 선물을 고를 때 두 고양이 중 누구에게 줄지 정하는 경우.

2) 순열(배치)
- 두 고양이를 한 줄로 나열하는 방법의 수: 2! = 2
- 예: 두 고양이가 앞뒤로 서 있는 순서를 생각할 때.

3) 간식 고르기
- 간식 종류가 S가지일 때, 두 고양이가 각각 하나씩 고르는 경우의 수
  - 중복 허용: S^2
  - 서로 다른 간식만 허용: S*(S-1)
- 예: 간식이 4가지면 중복 허용은 16가지, 서로 다르게 고르면 12가지

4) 몸무게 합/평균
- 두 고양이의 몸무게를 w1 kg, w2 kg라고 하자
  - 합: w1 + w2
  - 평균: (w1 + w2) / 2
- 예: w1 = 2.5 kg, w2 = 3.7 kg → 합 6.2 kg, 평균 3.1 kg

5) 간식 총량의 똑같은 분배
- 간식 총량이 T g일 때 두 고양이에게 똑같이 나누면 각 T/2 g

6) 비교·배분 응용
- 예를 들어 두 고양이 중 어느 쪽이 더 무거운지, 어느 쪽에게 더 많은 간식을 주는 것이 좋은지 등을 비교 문제로 만들 수 있음.

원하시는 문제 유형이나 실제 데이터(예: 두 고양이의 실제 몸무게, 간식 종류 수, 총 간식량 등)를 알려주시면 바로 계산해 드리겠습니다.


In [48]:
# 마지막 질문
# 문맥을 파악해야만 풀 수 있는 문제
response3 = chain_with_history.invoke(
    {'skill':'math', 'query':'철수랑 영희랑 합해서 몇 마리의 동물을 키우고 있나요??'},
    config={'configurable':{'session_id':'abc'}}
)

print(response3.content)

총 5마리입니다. 철수의 강아지 3마리 + 영희의 고양이 2마리 = 3 + 2 = 5. 
다른 데이터가 있으면 알려 주세요.


In [49]:
# 저장소 확인
print(store['abc'])

Human: 철수는 강아지를 세마리 키우고 있습니다.
AI: 도와드릴게요! 현재 문장은 상황 서술인데, 수학 문제로 활용할 수 있는 예를 몇 가지 바로 보여드릴게요. 원하시는 문제나 데이터를 주시면 바로 계산해 드리겠습니다.

예시 문제들
- 조합: 세 마리 중 두 마리를 골라 팀을 만든다면 경우의 수는?
  3C2 = 3가지

- 순열: 세 마리의 강아지를 한 줄로 나열하는 방법 수는?
  3! = 6가지

- 간식 선택 예시: 강아지 3마리가 서로 다른 간식을 고르는 경우의 수를 2종류로 가정하면?
  2^3 = 8가지

- 무게 합 예시: 세 강아지의 몸무게가 각각 4 kg, 5 kg, 6 kg일 때 전체 무게는?
  합 = 4 + 5 + 6 = 15 kg
  평균 무게 = 15 / 3 = 5 kg

- 비율/배분 예시: 세 강아지에게 간식 총량을 똑같이 나누면 각 강아지는 몇 그램이 받나? (총량이 주어지면 계산)

원하시는 문제 유형이나 실제 데이터(예: 각 강아지의 무게, 간식 종류 수 등)를 알려주시면 바로 계산해 드리겠습니다.
Human: 영회가 고양이를 두 마리 키우고 있습니다.
AI: 좋아요! 영회가 고양이 두 마리인 상황으로 바로 활용할 수 있는 수학 예시를 몇 가지 정리해 드릴게요.

예시 문제 및 해설
1) 조합
- 두 고양이 중 한 마리를 고르는 경우의 수: 2C1 = 2
- 예: 어떤 보호자 선물을 고를 때 두 고양이 중 누구에게 줄지 정하는 경우.

2) 순열(배치)
- 두 고양이를 한 줄로 나열하는 방법의 수: 2! = 2
- 예: 두 고양이가 앞뒤로 서 있는 순서를 생각할 때.

3) 간식 고르기
- 간식 종류가 S가지일 때, 두 고양이가 각각 하나씩 고르는 경우의 수
  - 중복 허용: S^2
  - 서로 다른 간식만 허용: S*(S-1)
- 예: 간식이 4가지면 중복 허용은 16가지, 서로 다르게 고르면 12가지

4) 몸무게 합/평균
- 두 고양이의 몸무게를 w1 kg, w2 kg라고 하자
  - 합: w1 + w2
  -